## Librerías y rutas

In [ ]:
!pip install pandas openpyxl numpy

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from openpyxl import load_workbook

# ── Rutas de entrada ──────────────────────────────────────────────────────────
PATH_V1      = "uam_tweets_2023_2026.csv"
PATH_V2      = "uam_tweets_v2.csv"
PATH_V3      = "uam_tweets_v3.csv"
PATH_REVIEWS = "Reseñas_UAM_google.xlsx"

# ── Rutas de salida ───────────────────────────────────────────────────────────
OUT_V1      = "tweets_v1_limpio.csv"
OUT_V2      = "tweets_v2_limpio.csv"
OUT_V3      = "tweets_v3_limpio.csv"
OUT_CORPUS  = "corpus_tweets_final.csv"
OUT_REVIEWS = "resenas_limpio.csv"

print("Configuración lista.")


---
## Filtros y funciones comunes a todos los corpus




In [ ]:
# ── Cuentas institucionales (identificadas en análisis exploratorio v1) ─────
CUENTAS_INST = {
    'UAM_Madrid', 'fuam_uam', 'UCCUAM', 'meaic_uam', 'CBM_CSIC_UAM',
    'EDoctorado_UAM', 'EUECREM_UAM', 'UAM_Biblioteca', 'EdinnovaUAM',
    'VEmerg_ExperUAM', 'GeografiaUam', 'AnaLopezUAM', 'Contemporan_UAM',
    'HistoriaUam', 'FyL_UAM', 'derecho_uam', 'CBMSO_CSIC_UAM',
    'isanidad', '24horas_rne', 'ondaceromn', 'SERMadridNorte',
    'Conversation_E', 'elpaiseducacion', 'FundacionLilly', 'fundacionsi',
    'ENAIRE', 'ONCE_oficial', 'LRicoti', 'EnsedeCiencia', 'FECYT_Ciencia',
    'EventoCiencia', 'UniCienciaSomos', 'RACiencias', 'OpositaTest',
    'Registrador_es', 'MusicBAcademy', 'Hospital_FJD', 'CopMadrid',
    # Cuentas UAM México detectadas en v2
    'uamxoficial', 'UAMIztapalapa', 'UAMAzcapotzalco', 'UAMXochimilco',
    'UAMLerma', 'UAMCuajimalpa',
}

# ── Patrón institucional por username ─────────────────────────────────────────
PATRON_INST = re.compile(
    r'(_uam$|^uam_|ministerio|rectorado|facultad|decanato|csic|hospital'
    r'|fundacion|foundation|academia|noticias|media|news|radio|tv'
    r'|ciencia|investig|official|oficial)', re.IGNORECASE
)

# ── Términos de contexto mexicano ─────────────────────────────────────────────
KW_MEXICO = [
    'uam xochimilco', 'uam iztapalapa', 'uam azcapotzalco',
    'uam lerma', 'uam cuajimalpa', 'correo.uam.mx', 'uam.mx',
    'casa abierta al tiempo', 'cdmx', 'ciudad de méxico',
    'estado de méxico', 'conacyt', 'unam', 'ipn',
]
PAT_MEXICO = re.compile('|'.join(re.escape(k) for k in KW_MEXICO), re.IGNORECASE)

# ── Falsos positivos v3: "Comunidad Autónoma de Madrid" ───────────────────────
# La ancla geográfica "Autónoma de Madrid" captura también tweets sobre
# la Comunidad Autónoma de Madrid (gobierno regional), que no son relevantes.
PAT_COMUNIDAD = re.compile(r'comunidad\s+aut[oó]noma\s+de\s+madrid', re.IGNORECASE)

# ── Función de longitud de texto real (sin URLs ni menciones) ─────────────────
def texto_len(text):
    t = re.sub(r'https?://\S+', '', str(text))
    t = re.sub(r'@\w+', '', t)
    return len(re.sub(r'\s+', ' ', t).strip())

# ── Función de normalización del texto ───────────────────────────────────────
def normalizar_texto(text):
    t = str(text)
    t = re.sub(r'https?://\S+', '', t)           # URLs
    t = re.sub(r'@\w+', '', t)                   # menciones
    t = re.sub(r'#(\w+)', r'\1', t)             # hashtags → palabra
    t = re.sub(r'[\r\n\t]+', ' ', t)           # saltos de línea
    t = re.sub(r'[^\w\s.,;:!?¡¿áéíóúüñÁÉÍÓÚÜÑ\'\"-]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t.lower()

print("Filtros y funciones comunes cargados.")


---
## Limpieza corpus v1 (`uam_tweets_2023_2026.csv`)




In [ ]:
df1 = pd.read_csv(PATH_V1)
print(f"V1 original: {len(df1)} tweets")

n = len(df1)
df1 = df1[df1['lang'] == 'es']
print(f"  Tras filtro idioma:               -{n - len(df1):>4}  → {len(df1)}")

n = len(df1); df1 = df1.drop_duplicates('tweet_id').drop_duplicates('text')
print(f"  Tras deduplicación:               -{n - len(df1):>4}  → {len(df1)}")

n = len(df1)
df1 = df1[~df1['author_username'].isin(CUENTAS_INST)]
df1 = df1[~df1['author_username'].str.contains(PATRON_INST, na=False)]
print(f"  Tras excluir institucionales:     -{n - len(df1):>4}  → {len(df1)}")

n = len(df1); df1['tl'] = df1['text'].apply(texto_len); df1 = df1[df1['tl'] >= 30]
print(f"  Tras filtro longitud (<30 chars): -{n - len(df1):>4}  → {len(df1)}")

df1['text_clean'] = df1['text'].apply(normalizar_texto)
df1['created_at_dt'] = pd.to_datetime(df1['created_at'], format='%a %b %d %H:%M:%S +0000 %Y', utc=True)
df1['fecha']   = df1['created_at_dt'].dt.date
df1['año']     = df1['created_at_dt'].dt.year
df1['mes']     = df1['created_at_dt'].dt.month
df1['año_mes'] = df1['created_at_dt'].dt.to_period('M').astype(str)
df1['version'] = 'v1'

print(f"\nV1 limpio: {len(df1)} tweets")


---
## Limpieza corpus v2 (`uam_tweets_v2.csv`)



In [ ]:
df2 = pd.read_csv(PATH_V2)
print(f"V2 original: {len(df2)} tweets")

n = len(df2)
df2 = df2[df2['lang'] == 'es']
print(f"  Tras filtro idioma:               -{n - len(df2):>4}  → {len(df2)}")

n = len(df2); df2 = df2.drop_duplicates('tweet_id').drop_duplicates('text')
print(f"  Tras deduplicación:               -{n - len(df2):>4}  → {len(df2)}")

n = len(df2)
df2 = df2[~df2['text'].apply(lambda t: bool(PAT_MEXICO.search(str(t))))]
print(f"  Tras filtro UAM México:           -{n - len(df2):>4}  → {len(df2)}")

n = len(df2); df2['tl'] = df2['text'].apply(texto_len); df2 = df2[df2['tl'] >= 30]
print(f"  Tras filtro longitud (<30 chars): -{n - len(df2):>4}  → {len(df2)}")

df2['text_clean'] = df2['text'].apply(normalizar_texto)
df2['created_at_dt'] = pd.to_datetime(df2['created_at'], format='%a %b %d %H:%M:%S +0000 %Y', utc=True)
df2['fecha']   = df2['created_at_dt'].dt.date
df2['año']     = df2['created_at_dt'].dt.year
df2['mes']     = df2['created_at_dt'].dt.month
df2['año_mes'] = df2['created_at_dt'].dt.to_period('M').astype(str)
df2['version'] = 'v2'

print(f"\nV2 limpio: {len(df2)} tweets")


---
## Limpieza corpus v3 (`uam_tweets_v3.csv`)




In [ ]:
df3 = pd.read_csv(PATH_V3)
print(f"V3 original: {len(df3)} tweets")

n = len(df3)
df3 = df3[df3['lang'] == 'es']
print(f"  Tras filtro idioma (catalán):     -{n - len(df3):>4}  → {len(df3)}")

n = len(df3); df3 = df3.drop_duplicates('tweet_id').drop_duplicates('text')
print(f"  Tras deduplicación:               -{n - len(df3):>4}  → {len(df3)}")

n = len(df3)
df3 = df3[~df3['text'].apply(lambda t: bool(PAT_COMUNIDAD.search(str(t))))]
print(f"  Tras filtro 'Comunidad Autónoma': -{n - len(df3):>4}  → {len(df3)}")

n = len(df3); df3['tl'] = df3['text'].apply(texto_len); df3 = df3[df3['tl'] >= 30]
print(f"  Tras filtro longitud (<30 chars): -{n - len(df3):>4}  → {len(df3)}")

df3['text_clean'] = df3['text'].apply(normalizar_texto)
df3['created_at_dt'] = pd.to_datetime(df3['created_at'], format='%a %b %d %H:%M:%S +0000 %Y', utc=True)
df3['fecha']   = df3['created_at_dt'].dt.date
df3['año']     = df3['created_at_dt'].dt.year
df3['mes']     = df3['created_at_dt'].dt.month
df3['año_mes'] = df3['created_at_dt'].dt.to_period('M').astype(str)
df3['version'] = 'v3'

print(f"\nV3 limpio: {len(df3)} tweets")


---
## 5. Combinación y deduplicación cruzada




In [ ]:
COLS = ['tweet_id', 'created_at', 'fecha', 'año', 'mes', 'año_mes',
        'text', 'text_clean', 'lang',
        'likes', 'replies', 'retweets', 'quotes', 'views',
        'author_username', 'author_name', 'url',
        'query_type', 'month', 'version']

combinado = pd.concat([df1[COLS], df2[COLS], df3[COLS]], ignore_index=True)

n = len(combinado)
combinado = combinado.drop_duplicates('tweet_id')
print(f"Eliminados por tweet_id duplicado (entre versiones): {n - len(combinado)}")

n = len(combinado)
combinado = combinado.drop_duplicates('text_clean')
print(f"Eliminados por texto duplicado (entre versiones):    {n - len(combinado)}")

combinado = combinado.reset_index(drop=True)
print(f"\nCorpus combinado final: {len(combinado)} tweets únicos")
print(f"\nPor versión de origen:")
print(combinado['version'].value_counts())
print(f"\nPor año:")
print(combinado['año'].value_counts().sort_index())
print(f"\nPor query_type:")
print(combinado['query_type'].value_counts())


## Estadísticas del corpus final

In [ ]:
print("=== Cobertura temporal ===")
print(f"Primer tweet: {combinado['fecha'].min()}")
print(f"Último tweet: {combinado['fecha'].max()}")
print(f"Meses cubiertos: {combinado['año_mes'].nunique()}")

print("\n=== Engagement ===")
print(combinado[['likes','replies','retweets','views']].describe().round(1))

print("\n=== Longitud media de texto (caracteres) ===")
print(f"  Original:  {combinado['text'].str.len().mean():.0f} chars")
print(f"  Limpio:    {combinado['text_clean'].str.len().mean():.0f} chars")

print("\n=== Tweets por año-mes ===")
print(combinado['año_mes'].value_counts().sort_index().to_string())


## Exportación del corpus de tweets

In [ ]:
# Datasets individuales por versión
df1[COLS].to_csv(OUT_V1, index=False, encoding='utf-8-sig')
df2[COLS].to_csv(OUT_V2, index=False, encoding='utf-8-sig')
df3[COLS].to_csv(OUT_V3, index=False, encoding='utf-8-sig')

# Corpus combinado final
combinado.to_csv(OUT_CORPUS, index=False, encoding='utf-8-sig')

print(f" tweets_v1_limpio.csv    → {len(df1)} tweets")
print(f" tweets_v2_limpio.csv    → {len(df2)} tweets")
print(f" tweets_v3_limpio.csv    → {len(df3)} tweets")
print(f" corpus_tweets_final.csv → {len(combinado)} tweets")


---
## Limpieza de reseñas Google (`Reseñas_UAM_google.xlsx`)



In [ ]:
wb = load_workbook(PATH_REVIEWS, read_only=True)
rows = list(wb.active.iter_rows(values_only=True))
COL_NAMES = ['autor','fuente','fuente_href','report_href',
             'puntuacion','fecha_rel','sep','texto','extra_href']
df_g = pd.DataFrame(rows[1:], columns=COL_NAMES)
print(f"Reseñas originales: {len(df_g)}")

n = len(df_g); df_g = df_g[df_g['fuente'] == 'Google']
print(f"  Tras filtro fuente (solo Google): -{n - len(df_g):>4}  → {len(df_g)}")

n = len(df_g); df_g = df_g[df_g['texto'].fillna('').str.strip().str.len() >= 10]
print(f"  Tras filtro texto vacío:          -{n - len(df_g):>4}  → {len(df_g)}")

def parsear_punt(val):
    if pd.isna(val): return np.nan
    m = re.match(r'(\d+(?:\.\d+)?)/(\d+)', str(val).strip())
    return round((float(m.group(1)) / float(m.group(2))) * 5, 2) if m else np.nan

df_g['puntuacion_5'] = df_g['puntuacion'].apply(parsear_punt)
df_g['sentimiento_proxy'] = df_g['puntuacion_5'].apply(
    lambda p: 'positivo' if p >= 4 else ('neutro' if p == 3 else 'negativo') if pd.notna(p) else np.nan
)

def normalizar_resena(text):
    t = re.sub(r'\.{2,}$', '', str(text).strip())
    t = re.sub(r'[\r\n\t]+', ' ', t)
    t = re.sub(r'[^\w\s.,;:!?¡¿áéíóúüñÁÉÍÓÚÜÑ\'\"-]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t.lower()

df_g['text_clean'] = df_g['texto'].apply(normalizar_resena)
n = len(df_g); df_g = df_g.drop_duplicates('text_clean')
print(f"  Tras deduplicación por texto:     -{n - len(df_g):>4}  → {len(df_g)}")

print(f"\nReseñas limpias: {len(df_g)}")
print(f"Distribución sentimiento:")
print(df_g['sentimiento_proxy'].value_counts())
print(f"Media puntuación: {df_g['puntuacion_5'].mean():.2f} / 5")


In [ ]:
COLS_RES = ['autor','puntuacion_5','sentimiento_proxy','fecha_rel','texto','text_clean']
df_g[COLS_RES].to_csv(OUT_REVIEWS, index=False, encoding='utf-8-sig')
print(f" resenas_limpio.csv → {len(df_g)} reseñas")
